# W&B Exports EDA and Comparison

This notebook analyzes the two W&B CSV exports in `wandb exports/`, compares model behavior across configurations, and saves additional comparison figures to `wandb exports/figures`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

cwd = Path.cwd()
if (cwd / "wandb exports").exists():
    EXPORT_DIR = cwd / "wandb exports"
elif cwd.name.lower() == "wandb exports":
    EXPORT_DIR = cwd
else:
    raise FileNotFoundError("Run this notebook from repo root or from 'wandb exports' directory.")

FIG_DIR = EXPORT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

csv_files = sorted(EXPORT_DIR.glob("wandb_export_*.csv"))
if len(csv_files) < 2:
    raise FileNotFoundError("Expected at least two wandb_export_*.csv files.")

print("Export dir:", EXPORT_DIR)
print("Figure dir:", FIG_DIR)
print("CSV files:")
for f in csv_files:
    print(" -", f.name)

In [ ]:
def load_and_classify_exports(files):
    dfs = [pd.read_csv(f) for f in files]
    fert = None
    crop = None

    for df in dfs:
        cols = set(df.columns)
        if {"nonadaptive", "total_years"}.issubset(cols):
            fert = df.copy()
        elif {"non_adaptive", "eval_det/mean_reward"}.issubset(cols):
            crop = df.copy()

    if fert is None or crop is None:
        dfs_sorted = sorted(dfs, key=len, reverse=True)
        fert = fert if fert is not None else dfs_sorted[0].copy()
        crop = crop if crop is not None else dfs_sorted[-1].copy()

    return fert, crop


def as_bool_label(series, true_label, false_label):
    s = series.astype(str).str.strip().str.lower()
    return s.map({"true": true_label, "false": false_label}).fillna("unknown")


fert_raw, crop_raw = load_and_classify_exports(csv_files)

print("Fertilization export shape:", fert_raw.shape)
print("Crop planning export shape:", crop_raw.shape)

## Data Quality Snapshot

In [ ]:
def quality_snapshot(df, name):
    out = pd.DataFrame({
        "rows": [len(df)],
        "cols": [df.shape[1]],
        "missing_cells": [int(df.isna().sum().sum())],
        "missing_pct": [float(df.isna().mean().mean() * 100.0)]
    }, index=[name])
    return out


quality = pd.concat([
    quality_snapshot(fert_raw, "fertilization"),
    quality_snapshot(crop_raw, "crop_planning")
])
display(quality)

if "State" in fert_raw.columns:
    print("\nFertilization run state distribution:")
    display(fert_raw["State"].value_counts(dropna=False).to_frame("count"))

if "State" in crop_raw.columns:
    print("\nCrop planning run state distribution:")
    display(crop_raw["State"].value_counts(dropna=False).to_frame("count"))

## Fertilization Experiment Analysis

In [ ]:
def prep_fertilization(df):
    d = df.copy()
    d["method"] = d.get("method", pd.Series(index=d.index, dtype=str)).astype(str)
    d["State"] = d.get("State", pd.Series(index=d.index, dtype=str)).astype(str)
    d["adaptive_label"] = as_bool_label(d["nonadaptive"], "nonadaptive", "adaptive") if "nonadaptive" in d.columns else "unknown"
    d["fixed_weather_label"] = as_bool_label(d["fixed_weather"], "fixed_weather", "random_weather") if "fixed_weather" in d.columns else "unknown"
    d["seed_num"] = pd.to_numeric(d.get("seed"), errors="coerce")
    d["total_years_num"] = pd.to_numeric(d.get("total_years"), errors="coerce")

    for col in [
        "eval_test_det/mean_reward",
        "eval_test_sto/mean_reward",
        "deterministic_return",
        "stochastic_return_mean",
        "pak_holdout_return",
    ]:
        if col in d.columns:
            d[col] = pd.to_numeric(d[col], errors="coerce")
    return d


fert = prep_fertilization(fert_raw)
fert_finished = fert[fert["State"] == "finished"].copy()
fert_metric = "eval_test_det/mean_reward" if "eval_test_det/mean_reward" in fert_finished.columns else None

print("Finished fertilization runs:", len(fert_finished))

if fert_metric is not None and len(fert_finished):
    fert_summary = (
        fert_finished
        .groupby(["method", "adaptive_label", "fixed_weather_label"], dropna=False)[fert_metric]
        .agg(["count", "mean", "std", "max"])
        .sort_values("mean", ascending=False)
    )
    display(fert_summary.head(20))

    winner = fert_finished.dropna(subset=[fert_metric]).copy()
    if "pak_holdout_return" in winner.columns:
        winner["pak_holdout_return"] = pd.to_numeric(winner["pak_holdout_return"], errors="coerce")
        winner["score"] = winner[fert_metric].rank(pct=True)
        if winner["pak_holdout_return"].notna().any():
            winner["score"] = winner["score"] + winner["pak_holdout_return"].rank(pct=True)
        winner = winner.sort_values("score", ascending=False)
    else:
        winner = winner.sort_values(fert_metric, ascending=False)

    winner_cols = [
        c for c in [
            "Name", "method", "adaptive_label", "fixed_weather_label", "total_years_num", "seed_num",
            fert_metric, "pak_holdout_return", "score"
        ] if c in winner.columns
    ]
    print("Top fertilization configs:")
    display(winner[winner_cols].head(10))
else:
    print("No finished fertilization runs or metric column not found.")

In [ ]:
if fert_metric is not None and len(fert_finished):
    plt.figure(figsize=(9, 5))
    sns.boxplot(data=fert_finished, x="method", y=fert_metric, hue="adaptive_label")
    plt.title("Fertilization: Deterministic Test Reward by Method and Policy Mode")
    plt.xlabel("Method")
    plt.ylabel(fert_metric)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_fert_method_boxplot.png", dpi=180)
    plt.show()

    agg = fert_finished.groupby(["adaptive_label", "fixed_weather_label"], dropna=False)[fert_metric].mean().unstack()
    if not agg.empty:
        plt.figure(figsize=(7, 4))
        sns.heatmap(agg, annot=True, fmt=".2f", cmap="YlGnBu")
        plt.title("Fertilization: Mean Deterministic Reward\nPolicy Mode vs Weather Mode")
        plt.xlabel("Weather Mode")
        plt.ylabel("Policy Mode")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "eda_fert_adaptive_weather_heatmap.png", dpi=180)
        plt.show()

    if "pak_holdout_return" in fert_finished.columns:
        sc = fert_finished.dropna(subset=[fert_metric, "pak_holdout_return"]).copy()
        if len(sc):
            plt.figure(figsize=(8, 5))
            sns.scatterplot(
                data=sc,
                x=fert_metric,
                y="pak_holdout_return",
                hue="adaptive_label",
                style="fixed_weather_label",
                size="total_years_num",
                sizes=(40, 220),
                alpha=0.85,
            )
            plt.title("Fertilization: In-Sample vs Pakistan Holdout")
            plt.tight_layout()
            plt.savefig(FIG_DIR / "eda_fert_insample_vs_holdout.png", dpi=180)
            plt.show()
else:
    print("Skipping fertilization plots due to missing finished runs/metric.")

## Crop Planning Experiment Analysis

In [ ]:
def prep_crop(df):
    d = df.copy()
    d["method"] = d.get("method", pd.Series(index=d.index, dtype=str)).astype(str)
    d["State"] = d.get("State", pd.Series(index=d.index, dtype=str)).astype(str)
    d["adaptive_label"] = as_bool_label(d["non_adaptive"], "nonadaptive", "adaptive") if "non_adaptive" in d.columns else "unknown"
    d["fixed_weather_label"] = as_bool_label(d["fixed_weather"], "fixed_weather", "random_weather") if "fixed_weather" in d.columns else "unknown"
    d["seed_num"] = pd.to_numeric(d.get("seed"), errors="coerce")

    metric_cols = [c for c in d.columns if "mean_reward" in c]
    for c in metric_cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d


crop = prep_crop(crop_raw)
crop_finished = crop[crop["State"] == "finished"].copy()
crop_metric = "eval_det/mean_reward" if "eval_det/mean_reward" in crop_finished.columns else None

print("Finished crop planning runs:", len(crop_finished))

if crop_metric is not None and len(crop_finished):
    crop_summary = (
        crop_finished
        .groupby(["method", "adaptive_label", "fixed_weather_label"], dropna=False)[crop_metric]
        .agg(["count", "mean", "std", "max"])
        .sort_values("mean", ascending=False)
    )
    display(crop_summary)

    winner = crop_finished.dropna(subset=[crop_metric]).copy()
    score_cols = [
        c for c in [
            "eval_det/mean_reward",
            "eval_sto/mean_reward",
            "eval_det_new_years/mean_reward",
            "eval_det_other_loc/mean_reward",
        ] if c in winner.columns
    ]
    winner["score"] = 0.0
    for c in score_cols:
        if winner[c].notna().any():
            winner["score"] += winner[c].rank(pct=True)
    winner = winner.sort_values("score", ascending=False)

    winner_cols = [
        c for c in [
            "Name", "method", "adaptive_label", "fixed_weather_label", "seed_num",
            "eval_det/mean_reward", "eval_sto/mean_reward",
            "eval_det_new_years/mean_reward", "eval_det_other_loc/mean_reward", "score"
        ] if c in winner.columns
    ]
    print("Top crop planning configs:")
    display(winner[winner_cols].head(10))
else:
    print("No finished crop planning runs or metric column not found.")

In [ ]:
if crop_metric is not None and len(crop_finished):
    plt.figure(figsize=(10, 5))
    sns.barplot(
        data=crop_finished,
        x="method",
        y=crop_metric,
        hue="fixed_weather_label",
        estimator="mean",
        errorbar="sd",
    )
    plt.title("Crop Planning: Deterministic Eval Reward by Method and Weather")
    plt.xlabel("Method")
    plt.ylabel(crop_metric)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "eda_crop_method_weather_bar.png", dpi=180)
    plt.show()

    agg = crop_finished.groupby(["adaptive_label", "fixed_weather_label"], dropna=False)[crop_metric].mean().unstack()
    if not agg.empty:
        plt.figure(figsize=(7, 4))
        sns.heatmap(agg, annot=True, fmt=".2f", cmap="OrRd")
        plt.title("Crop Planning: Mean Deterministic Reward\nPolicy Mode vs Weather Mode")
        plt.xlabel("Weather Mode")
        plt.ylabel("Policy Mode")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "eda_crop_adaptive_weather_heatmap.png", dpi=180)
        plt.show()

    eval_cols = [
        "eval_det/mean_reward",
        "eval_sto/mean_reward",
        "eval_det_new_years/mean_reward",
        "eval_sto_new_years/mean_reward",
        "eval_det_other_loc/mean_reward",
        "eval_sto_other_loc/mean_reward",
    ]
    available = [c for c in eval_cols if c in crop_finished.columns]
    if available:
        long_df = crop_finished.melt(
            id_vars=["method", "adaptive_label", "fixed_weather_label", "Name"],
            value_vars=available,
            var_name="metric",
            value_name="value",
        ).dropna(subset=["value"])

        plt.figure(figsize=(12, 5))
        sns.barplot(data=long_df, x="metric", y="value", hue="method", estimator="mean", errorbar=None)
        plt.title("Crop Planning: Metric-Level Comparison by Method")
        plt.xlabel("Metric")
        plt.ylabel("Mean reward")
        plt.xticks(rotation=35, ha="right")
        plt.tight_layout()
        plt.savefig(FIG_DIR / "eda_crop_metric_panel.png", dpi=180)
        plt.show()
else:
    print("Skipping crop planning plots due to missing finished runs/metric.")

## Practical Readout for Thesis Story

In [ ]:
def completion_rate(df):
    if "State" not in df.columns or len(df) == 0:
        return np.nan
    return float((df["State"].astype(str) == "finished").mean())


fert_completion = completion_rate(fert)
crop_completion = completion_rate(crop)

print("Completion rate (fertilization):", f"{fert_completion:.1%}" if pd.notna(fert_completion) else "n/a")
print("Completion rate (crop planning):", f"{crop_completion:.1%}" if pd.notna(crop_completion) else "n/a")

print("\nRecommendation framing:")
print("1) Use top finished configuration(s) as inference preset for demo/pilot.")
print("2) Retraining is only required if target region/crop/weather regime differs materially from training distribution.")
print("3) For Pakistani deployment, prioritize configs with both high test reward and strong Pakistan holdout behavior.")
print("4) Keep periodic re-training optional (e.g., seasonal or annual) as new local data arrives.")

print("\nSaved EDA figures in:", FIG_DIR)